# ---------- Quais fatores influenciam um paciente a faltar à consulta? ----------

### 1. Conhecendo os dados

In [195]:
import pandas as pd
import numpy as np
import plotly.express as px

In [196]:
df = pd.read_csv("Medical_Appointment_No_Shows.csv")

In [197]:
display(df.head())

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [198]:
df["No-show"] = df["No-show"].str.replace("Yes", "Sim").str.replace("No", "Não")
df["Gender"] = df["Gender"].str.replace("M", "Homem").str.replace("F", "Mulher")

### 2. Limpeza e tratamento dos dados

In [199]:
qtde_dados_vazios = df.isna().sum()
print(f"Qtde de dados vazios: \n{qtde_dados_vazios}")

Qtde de dados vazios: 
PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0
dtype: int64


In [200]:
qtde_dados_duplicados = df.duplicated().sum()
print(f"Qtde de linhas duplicados: {qtde_dados_duplicados}")

Qtde de linhas duplicados: 0


In [201]:
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  str    
 3   ScheduledDay    110527 non-null  str    
 4   AppointmentDay  110527 non-null  str    
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  str    
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  str    
dtypes: float64(1), int64(8), str(5)
memory usage: 18.3 MB


None

In [202]:
df["ScheduledDay"] = pd.to_datetime(df["ScheduledDay"])
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"])

### 3. Estatísticas gerais

In [203]:
qtde_de_clientes = df["PatientId"].count()
idade_media = df["Age"].mean()

print(f"Qtde de Clientes: {qtde_de_clientes}")
print(f"Idade Média dos Clientes: {idade_media:.2f} anos")

Qtde de Clientes: 110527
Idade Média dos Clientes: 37.09 anos


### 4. Quantas pessoas faltaram?

In [204]:
quantos_faltaram = df["No-show"].value_counts().reset_index()

graf1 = px.pie(quantos_faltaram, 
               values="count", 
               names="No-show", 
               labels={'No-show': 'Apareceram?', 'count': 'Quantidade'},
               color="No-show",
               color_discrete_map={"Não": "#d62728", "Sim": "#2ca02c"}
            )               
    
graf1.update_layout(paper_bgcolor="#282C34", font_color="white")

graf1.show()

### 5. Analise dos Dados

##### Gênero influência?

In [205]:
genero = pd.crosstab(df["Gender"], df["No-show"], normalize="index") * 100

graf2 = px.bar(genero, 
               barmode='group',
               labels={"Gender": "Gênero", "No-show": "Apareceram?"},
               color_discrete_map={"Sim":'#2ca02c',"Não": '#d62728'})

graf2.update_traces(texttemplate='%{y:,.1f}%', textposition='outside')

graf2.update_layout(
    paper_bgcolor="#282C34", 
    plot_bgcolor="#282C34", 
    font_color="white",
    yaxis=dict(title="", showticklabels=False, showgrid=False, visible=False),
    xaxis=dict(title="")
    )

graf2.show()

##### Idade influência?

In [220]:
df['Age_Range'] = pd.cut(df['Age'],
    bins=[0, 5, 18, 30, 45, 60, 120],
    labels=['0-4', '5-17', '18-29', '30-44', '45-59', '60+']
    )

idade = (pd.crosstab(df['Age_Range'], df['No-show'], normalize='index') * 100).round(1)

graf7 = px.bar(
    idade.reset_index(),
    x='Age_Range',
    y='Não',
    text='Não',
    labels={'Age_Range': 'Faixa Etária', 'Não': 'Taxa de Faltas (%)'},
    color='Não',
    color_continuous_scale='Reds'
)

graf7.update_traces(
    texttemplate='%{y:.1f}%',
    textposition='outside'
)

graf7.update_layout(
    title='Taxa de Faltas por Faixa Etária',
    paper_bgcolor='#282C34',
    plot_bgcolor='#282C34',
    font_color='white',
    xaxis=dict(title='Faixa Etária', showgrid=False),
    yaxis=dict(title='Taxa de Faltas (%)', showgrid=False),
    coloraxis_showscale=False
)

graf7.show()

##### SMS influência?

In [207]:
sms = pd.crosstab(df["SMS_received"], df["No-show"], normalize="index").rename(index={0: 'Não Recebeu SMS', 1: 'Recebeu SMS'}) * 100

graf3 = px.bar(sms, 
               barmode='group',
               labels={"SMS_received": "", "No-show": "Apareceram?"},
               color_discrete_map={"Sim":'#2ca02c',"Não": '#d62728'})

graf3.update_traces(texttemplate='%{y:,.1f}%', textposition='outside')

graf3.update_layout(
    paper_bgcolor="#282C34", 
    plot_bgcolor="#282C34", 
    font_color="white",
    yaxis=dict(title="", showticklabels=False, showgrid=False, visible=False),
    xaxis=dict(title=""))

graf3.show()

##### Hipertensão influencia?

In [208]:
hipertensao = pd.crosstab(df["Hipertension"], df["No-show"], normalize="index").rename(index={0: 'Não tem Hipertensão', 1: 'Tem Hipertensão'}) * 100

graf4 = px.bar(hipertensao, 
               barmode='group',
               labels={"Hipertension": "", "No-show": "Apareceram?"},
               color_discrete_map={"Sim":'#2ca02c',"Não": '#d62728'})

graf4.update_traces(texttemplate='%{y:,.1f}%', textposition='outside')

graf4.update_layout(
    paper_bgcolor="#282C34", 
    plot_bgcolor="#282C34", 
    font_color="white",
    yaxis=dict(title="", showticklabels=False, showgrid=False, visible=False),
    xaxis=dict(title=""))

graf4.show()

##### Diabetes influencia?

In [209]:
diabetes = pd.crosstab(df["Diabetes"], df["No-show"], normalize="index").rename(index={0: 'Não tem diabetes', 1: 'Tem diabetes'}) * 100

graf5 = px.bar(diabetes, 
               barmode='group',
               labels={"Diabetes": "", "No-show": "Apareceram?"},
               color_discrete_map={"Sim":'#2ca02c',"Não": '#d62728'})

graf5.update_traces(texttemplate='%{y:,.1f}%', textposition='outside')

graf5.update_layout(
    paper_bgcolor="#282C34", 
    plot_bgcolor="#282C34", 
    font_color="white",
    yaxis=dict(title="", showticklabels=False, showgrid=False, visible=False),
    xaxis=dict(title=""))

graf5.show()

##### Programa social influencia?

In [210]:
prog = pd.crosstab(df["Scholarship"], df["No-show"], normalize="index").rename(index={0: 'Não participa de programa social', 1: 'Participa de programa social'}) * 100

graf6 = px.bar(prog, 
               barmode='group',
               labels={"Scholarship": "", "No-show": "Apareceram?"},
               color_discrete_map={"Sim":'#2ca02c',"Não": '#d62728'})

graf6.update_traces(texttemplate='%{y:,.1f}%', textposition='outside')

graf6.update_layout(
    paper_bgcolor="#282C34", 
    plot_bgcolor="#282C34", 
    font_color="white",
    yaxis=dict(title="", showticklabels=False, showgrid=False, visible=False),
    xaxis=dict(title=""))

graf6.show()

##### Tempo de espera influencia?

In [221]:
df['Waiting_Time'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days 

df['Waiting_Range'] = pd.cut(df['Waiting_Time'],
    bins=[0, 5, 10, 20, 30, 60, 180],
    labels=['0-5', '6-10', '11-20', '21-30', '31-60', '60+']
)

espera_pct = (pd.crosstab(df['Waiting_Range'], df['No-show'], normalize='index') * 100).round(1)

graf8 = px.bar(
    espera_pct.reset_index(),
    x='Waiting_Range',
    y='Não',
    labels={"Waiting_Range": "Faixa_Etária"},
    text='Não',
    color='Não',
    color_continuous_scale='Reds'
)

graf8.update_traces(
    texttemplate='%{y:.1f}%',
    textposition='outside'
)

graf8.update_layout(
    title='Taxa de Faltas por Faixa de Espera',
    paper_bgcolor='#282C34',
    plot_bgcolor='#282C34',
    font_color='white',
    xaxis=dict(
        title='Tempo de Espera (Dias)',
        showgrid=False
    ),
    yaxis=dict(
        title='Taxa de Faltas (%)',
        showgrid=False
    ),
    coloraxis_showscale=False
)

graf8.show()

### 6. Conclusão

- O gênero do paciente, o recebimento de SMS e a presença de diabetes apresentaram diferenças muito pequenas nas taxas de ausência. Dessa forma, não foram observados indícios relevantes de associação entre essas variáveis e o não comparecimento às consultas.

- Pacientes com hipertensão apresentaram uma taxa de ausência 6,3 pontos percentuais superior à dos pacientes sem hipertensão. Essa diferença sugere uma associação moderada entre a hipertensão e as faltas às consultas.

- Pacientes participantes de programas sociais apresentaram uma diferença de 7,1 pontos percentuais na taxa de ausência em relação aos demais pacientes. Esse resultado sugere uma associação moderada entre a participação em programas sociais e o comparecimento às consultas.

- A análise por faixa etária indicou diferenças relevantes nas taxas de ausência entre os grupos, sugerindo que a idade pode estar associada ao comparecimento às consultas, visto que pacientes acima de 60 anos apresentaram quase 80% de faltas.

- A análise do tempo de espera mostrou diferenças relevantes nas taxas de ausência entre os grupos analisados, indicando que o intervalo entre o agendamento e a consulta pode estar associado ao comportamento de comparecimento dos pacientes; nota-se que pacientes com 0 a 5 dias de tempo de espera apresentam uma diferença de 10,1 pontos percentuais em relação aos pacientes com 31 a 60 dias de tempo de espera.